In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Any

import mne
import numpy as np
import pandas as pd


BDF_DIR: Path = Path("../dataset/raw/bdf")

In [2]:
def normalize_v1_subtract_1638144(event_codes: np.ndarray) -> np.ndarray:
    """
    Normalización usada inicialmente:
    1638147 -> 3
    """
    codes: np.ndarray = event_codes.copy().astype(int)
    mask: np.ndarray = codes >= 1638144
    codes[mask] = codes[mask] - 1638144
    return codes


def normalize_v2_mod_65536(event_codes: np.ndarray) -> np.ndarray:
    """
    Normalización basada en módulo:
    4194307 % 65536 -> 3
    """
    codes: np.ndarray = event_codes.copy().astype(int)
    mask: np.ndarray = codes >= 65536
    codes[mask] = codes[mask] % 65536
    return codes


def count_codes(event_codes: np.ndarray) -> Dict[int, int]:
    """
    Cuenta frecuencias de códigos de eventos.
    """
    unique_codes: np.ndarray
    counts: np.ndarray

    unique_codes, counts = np.unique(
        event_codes.astype(int),
        return_counts=True,
    )

    return dict(zip(unique_codes.tolist(), counts.tolist()))


def count_valid_345_sequences(event_codes: np.ndarray) -> int:
    """
    Cuenta cuántas secuencias 3->4->5 aparecen consecutivamente.
    """
    total: int = 0
    index: int = 0

    while index <= len(event_codes) - 3:
        window: list[int] = [
            int(event_codes[index]),
            int(event_codes[index + 1]),
            int(event_codes[index + 2]),
        ]

        if window == [3, 4, 5]:
            total += 1
            index += 3
        else:
            index += 1

    return total

In [3]:
def inspect_subject_events(subject_id: int) -> List[Dict[str, Any]]:
    """
    Inspecciona los últimos canales de un participante,
    probando eventos crudos y normalizados.
    """
    bdf_path: Path = BDF_DIR / f"s{subject_id:02d}.bdf"

    raw = mne.io.read_raw_bdf(
        bdf_path,
        preload=False,
        verbose=False,
    )

    rows: List[Dict[str, Any]] = []

    for channel_name in raw.ch_names[-5:]:
        try:
            events: np.ndarray = mne.find_events(
                raw,
                stim_channel=channel_name,
                shortest_event=1,
                verbose=False,
            )

            if events.size == 0:
                rows.append({
                    "subject": subject_id,
                    "channel": repr(channel_name),
                    "num_channels": len(raw.ch_names),
                    "events_found": 0,
                    "raw_counts": {},
                    "v1_counts": {},
                    "v2_counts": {},
                    "v1_relevant_count": 0,
                    "v2_relevant_count": 0,
                    "v1_valid_345_sequences": 0,
                    "v2_valid_345_sequences": 0,
                    "first_20_v1_relevant": [],
                    "first_20_v2_relevant": [],
                })
                continue

            raw_codes: np.ndarray = events[:, 2].astype(int)

            v1_codes: np.ndarray = normalize_v1_subtract_1638144(raw_codes)
            v2_codes: np.ndarray = normalize_v2_mod_65536(raw_codes)

            v1_relevant: np.ndarray = v1_codes[
                np.isin(v1_codes, [3, 4, 5])
            ]

            v2_relevant: np.ndarray = v2_codes[
                np.isin(v2_codes, [3, 4, 5])
            ]

            rows.append({
                "subject": subject_id,
                "channel": repr(channel_name),
                "num_channels": len(raw.ch_names),
                "events_found": len(events),
                "raw_counts": count_codes(raw_codes),
                "v1_counts": count_codes(v1_codes),
                "v2_counts": count_codes(v2_codes),
                "v1_relevant_count": len(v1_relevant),
                "v2_relevant_count": len(v2_relevant),
                "v1_valid_345_sequences": count_valid_345_sequences(v1_relevant),
                "v2_valid_345_sequences": count_valid_345_sequences(v2_relevant),
                "first_20_v1_relevant": v1_relevant[:20].astype(int).tolist(),
                "first_20_v2_relevant": v2_relevant[:20].astype(int).tolist(),
            })

        except Exception as error:
            rows.append({
                "subject": subject_id,
                "channel": repr(channel_name),
                "num_channels": len(raw.ch_names),
                "events_found": -1,
                "error": str(error),
                "raw_counts": {},
                "v1_counts": {},
                "v2_counts": {},
                "v1_relevant_count": 0,
                "v2_relevant_count": 0,
                "v1_valid_345_sequences": 0,
                "v2_valid_345_sequences": 0,
                "first_20_v1_relevant": [],
                "first_20_v2_relevant": [],
            })

    return rows

In [4]:
all_rows: List[Dict[str, Any]] = []

for subject_id in range(1, 33):
    print(f"Inspecting S{subject_id:02d}...")
    all_rows.extend(inspect_subject_events(subject_id))

events_df: pd.DataFrame = pd.DataFrame(all_rows)

events_df.head()

Inspecting S01...


/tmp/ipykernel_31746/896919027.py:18: RuntimeWarning: Trigger channel contains negative values, using absolute value. If data were acquired on a Neuromag system with STI016 active, consider using uint_cast=True to work around an acquisition bug
  events: np.ndarray = mne.find_events(


Inspecting S02...
Inspecting S03...
Inspecting S04...
Inspecting S05...
Inspecting S06...
Inspecting S07...
Inspecting S08...
Inspecting S09...
Inspecting S10...
Inspecting S11...
Inspecting S12...
Inspecting S13...
Inspecting S14...
Inspecting S15...
Inspecting S16...
Inspecting S17...
Inspecting S18...
Inspecting S19...
Inspecting S20...
Inspecting S21...
Inspecting S22...
Inspecting S23...
Inspecting S24...


/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(
/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(


Inspecting S25...


...

Inspecting S31...


/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channel names are not unique, found duplicates for: {''}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_bdf(
/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(
/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(


Inspecting S32...


/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channel names are not unique, found duplicates for: {''}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_bdf(
/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(
/tmp/ipykernel_31746/896919027.py:8: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(


,subject,channel,num_channels,events_found,raw_counts,v1_counts,v2_counts,v1_relevant_count,v2_relevant_count,v1_valid_345_sequences,v2_valid_345_sequences,first_20_v1_relevant,first_20_v2_relevant
0,1,'Erg2',48,0,{},{},{},0,0,0,0,[],[]
1,1,'Resp',48,0,{},{},{},0,0,0,0,[],[]
2,1,'Plet',48,16,"{1: 1, 2: 15}","{1: 1, 2: 15}","{1: 1, 2: 15}",0,0,0,0,[],[]
3,1,'Temp',48,24,{34: 24},{34: 24},{34: 24},0,0,0,0,[],[]
4,1,'Status',48,13760,"{1: 162, 2: 2, 3: 40, 4: 40, 5: 40, 6: 13475, ...","{1: 162, 2: 2, 3: 40, 4: 40, 5: 40, 6: 13475, ...","{1: 162, 2: 2, 3: 40, 4: 40, 5: 40, 6: 13475, ...",120,120,40,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."


In [5]:
summary_df: pd.DataFrame = events_df.sort_values(
    by=[
        "subject",
        "v2_valid_345_sequences",
        "v1_valid_345_sequences",
        "v2_relevant_count",
        "v1_relevant_count",
    ],
    ascending=[True, False, False, False, False],
)

best_candidates_df: pd.DataFrame = (
    summary_df
    .groupby("subject")
    .head(1)
    .reset_index(drop=True)
)

best_candidates_df[
    [
        "subject",
        "num_channels",
        "channel",
        "events_found",
        "v1_relevant_count",
        "v1_valid_345_sequences",
        "v2_relevant_count",
        "v2_valid_345_sequences",
        "first_20_v1_relevant",
        "first_20_v2_relevant",
    ]
]

,subject,num_channels,channel,events_found,v1_relevant_count,v1_valid_345_sequences,v2_relevant_count,v2_valid_345_sequences,first_20_v1_relevant,first_20_v2_relevant
0,1,48,'Status',13760,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
1,2,48,'Status',287,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
2,3,48,'Status',284,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
3,4,48,'Status',286,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
4,5,48,'Status',288,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
5,6,48,'Status',288,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
6,7,48,'Status',288,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
7,8,48,'Status',288,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
8,9,48,'Status',288,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."
9,10,48,'Status',288,120,40,120,40,"[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...","[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ..."


In [6]:
problematic_df: pd.DataFrame = best_candidates_df[
    (best_candidates_df["v2_valid_345_sequences"] != 40)
    & (best_candidates_df["v1_valid_345_sequences"] != 40)
]

problematic_df[
    [
        "subject",
        "channel",
        "raw_counts",
        "v1_counts",
        "v2_counts",
        "v1_valid_345_sequences",
        "v2_valid_345_sequences",
        "first_20_v1_relevant",
        "first_20_v2_relevant",
    ]
]

,subject,channel,raw_counts,v1_counts,v2_counts,v1_valid_345_sequences,v2_valid_345_sequences,first_20_v1_relevant,first_20_v2_relevant
27,28,'',"{1638145: 157, 1638146: 4, 1638147: 39, 163814...","{1: 157, 2: 4, 3: 39, 4: 37, 5: 39, 7: 1, 4194...","{65280: 1, 65281: 168, 65282: 4, 65283: 41, 65...",37,0,"[3, 5, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, ...",[]


In [7]:
output_path: Path = Path("../dataset/processed/status_event_diagnostic.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

events_df.to_csv(output_path, index=False)

print(f"Saved diagnostic to: {output_path}")

Saved diagnostic to: ../dataset/processed/status_event_diagnostic.csv
